# Notebook 05 — Extracción LLM sobre candidatos priorizados

A partir del dataset `candidates_priority.parquet` generado en el notebook 04, este notebook realiza la extracción asistida con LLM de variables relacionadas con la actividad de pesca por vídeo. La estrategia sigue siendo trazable y modular: se trabaja por lotes, se construyen prompts explícitos, se parsean respuestas estructuradas en JSON y se guardan resultados intermedios para poder reanudar el proceso sin repetir trabajo ya completado.

En esta nueva versión, el notebook ya no parte de un subconjunto extremadamente estricto, sino de un conjunto priorizado más amplio. Esto permite aumentar la cobertura del análisis y mantener coherencia con el nuevo enfoque del TFG, que busca reducir la dependencia de filtros manuales muy agresivos y aprovechar más información observacional procedente de YouTube.

## Objetivo

- Cargar el conjunto `candidates_priority` generado en el notebook 04.
- Construir un texto de entrada utilizable para el LLM a partir de título, descripción y, cuando exista, otras fuentes textuales asociadas.
- Definir el prompt y las funciones auxiliares de soporte para la extracción.
- Procesar la extracción en lotes pequeños y trazables.
- Persistir cada lote de resultados en archivos intermedios.
- Consolidar todos los lotes en un único dataset final.

## Variables extraídas

Las variables extraídas deben ser interpretables y útiles para las siguientes fases del pipeline. Entre ellas se incluyen:

- `activity_mentions_llm`: indicios textuales de actividad de pesca observada.
- `captures_count_llm`: número de capturas explícitas identificadas en el texto, si aparecen.
- `species_detected_llm`: especies o modalidades detectadas en el contenido textual.
- `activity_level_llm`: nivel cualitativo preliminar de actividad (`low`, `medium`, `high`).
- `certainty_llm`: grado de certeza de la extracción (`high`, `medium`, `low`).
- `evidence_llm`: fragmento textual breve que justifica la salida.
- `notes_llm`: observaciones adicionales del modelo sobre ambigüedad o falta de información.

## Entradas y salidas

**Entrada:** `outputs/llm_activity/candidates_priority.parquet`

**Salidas:**
- archivos intermedios por lote: `llm_priority_batch_*.parquet`
- consolidado final: `llm_extractions.parquet`
- exportación auxiliar: `llm_extractions.csv`

## Preparación del entorno

Se montan Google Drive, las rutas de entrada y salida y las librerías necesarias para ejecutar la extracción LLM de forma reproducible.

In [1]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import pandas as pd
import numpy as np
import json
import re
from ast import literal_eval

BASE_DIR = Path("/content/drive/MyDrive/PIDS4jjj2")

INPUT_PATH = BASE_DIR / "outputs" / "llm_activity" / "candidates_priority.parquet"
OUTPUT_DIR = BASE_DIR / "outputs" / "llm_activity"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FINAL_PARQUET_PATH = OUTPUT_DIR / "llm_extractions.parquet"
FINAL_CSV_PATH = OUTPUT_DIR / "llm_extractions.csv"

print("INPUT_PATH:", INPUT_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)

Mounted at /content/drive
INPUT_PATH: /content/drive/MyDrive/PIDS4jjj2/outputs/llm_activity/candidates_priority.parquet
OUTPUT_DIR: /content/drive/MyDrive/PIDS4jjj2/outputs/llm_activity


## Carga del conjunto priorizado

Se carga el subconjunto `candidates_priority`, que representa una versión priorizada y todavía suficientemente amplia del conjunto maestro. Este será el punto de partida para la extracción LLM.

In [2]:
df = pd.read_parquet(INPUT_PATH)

print("Shape candidates_priority:", df.shape)
df.head(3)

Shape candidates_priority: (151, 38)


,video_id,title,description,channel,channel_id,uploader,published_at,year,month,quarter,...,has_lucioperca_terms,has_spinning_terms,has_species_or_modality_terms,candidate_score,is_text_too_short,is_title_too_short,has_weak_context,has_no_valmayor,is_very_short_video,priority_score
0,K2oTCf-5C4c,rubenylolo......carpfishing 🤍🩵,#Carpfishing #otoñal #fishing #pesca #pescaria...,RubenYlolo,UCdv-Xd3cLlJT0CF9kdmZhcw,RubenYlolo,2026-04-30,2026,4,2,...,False,True,True,8,False,False,False,False,False,21
1,KYkoHio90Pw,Carpfishing 🥵 🔥Summer Campaign 🔥🥵 HOT FISHING🎣,CARPFISHING🥵 SUMMER🥵 CAMPAIGN 🔥🎣\n#pesca #fish...,Gerar Carp_angler,UC1z_zXQcdl6jqFYnMozTatg,Gerar Carp_angler,2026-02-25,2026,2,1,...,True,False,True,8,False,False,False,False,False,18
2,TvT1jlG9ayw,El embalse de Valmayor y su entorno,Visita nuestra web: http://www.infosierrademad...,Infosierrademadrid.es,UC62wX016CjABYJSHeP2RppQ,Infosierrademadrid.es,2016-07-18,2016,7,3,...,False,False,True,8,False,False,False,False,False,18


In [3]:
cols_to_show = [
    "video_id", "title", "channel", "year", "year_quarter",
    "priority_score", "has_valmayor", "has_fishing_terms",
    "has_carp_terms", "has_bass_terms", "has_lucio_terms",
    "has_lucioperca_terms", "has_spinning_terms"
]

existing_cols = [c for c in cols_to_show if c in df.columns]
df[existing_cols].head(10)

,video_id,title,channel,year,year_quarter,priority_score,has_valmayor,has_fishing_terms,has_carp_terms,has_bass_terms,has_lucio_terms,has_lucioperca_terms,has_spinning_terms
0,K2oTCf-5C4c,rubenylolo......carpfishing 🤍🩵,RubenYlolo,2026,2026-Q2,21,True,True,True,True,True,False,True
1,KYkoHio90Pw,Carpfishing 🥵 🔥Summer Campaign 🔥🥵 HOT FISHING🎣,Gerar Carp_angler,2026,2026-Q1,18,True,True,True,False,False,True,False
2,TvT1jlG9ayw,El embalse de Valmayor y su entorno,Infosierrademadrid.es,2016,2016-Q3,18,True,True,True,False,True,False,False
3,kt78DbVibg0,¡LLUVIA y MUCHAS CAPTURAS! 💥 Carp Fishing bajo...,RubenYlolo,2025,2025-Q4,18,True,True,True,False,True,False,False
4,dZfMo3t8TWo,🔥Sesion increíble!! 🔥en solitario en valma con...,RubenYlolo,2025,2025-Q4,18,True,True,True,False,True,False,False
5,uTM2DGKH-xw,Review Teflon Spare Spools [MV Spools],MvSpools EN,2021,2021-Q2,16,True,True,True,False,False,False,False
6,Yt-FrYfIfm4,rubenylolo.....valmayor,RubenYlolo,2026,2026-Q1,16,True,True,True,False,False,False,False
7,97L3xUDkpxQ,2ª Jornada de Carpfishing Valmayor,Carpfishing Valmayor,2014,2014-Q2,16,True,True,True,False,False,False,False
8,DrBaZj0cvLw,rubenylolo......by spinnigcarp,RubenYlolo,2026,2026-Q1,16,True,True,True,False,False,False,False
9,5daoPKGBVpI,Jornada de CARPFISHING VALMAYOR con mas de 20 ...,Carpfishing Valmayor,2015,2015-Q1,16,True,True,True,False,False,False,False


## Construcción del texto de entrada para el LLM

El LLM necesita una representación textual compacta y consistente por vídeo. Para ello se construye una columna `text_for_llm` que combina la información más relevante disponible, principalmente título y descripción. Si en versiones posteriores del pipeline se incorporan otras fuentes textuales, esta misma estructura podrá reutilizarse fácilmente.

In [4]:
for col in ["title", "description", "channel"]:
    if col in df.columns:
        df[col] = df[col].fillna("").astype(str)

df["text_for_llm"] = (
    "TITLE: " + df["title"].fillna("") + "\n\n" +
    "DESCRIPTION: " + df["description"].fillna("")
).str.strip()

df["text_len_llm"] = df["text_for_llm"].str.len()

print("Longitud media text_for_llm:", round(df["text_len_llm"].mean(), 2))
print("Longitud mediana text_for_llm:", round(df["text_len_llm"].median(), 2))

Longitud media text_for_llm: 697.64
Longitud mediana text_for_llm: 256.0


In [5]:
df[["video_id", "title", "year", "text_len_llm", "text_for_llm"]].head(5)

,video_id,title,year,text_len_llm,text_for_llm
0,K2oTCf-5C4c,rubenylolo......carpfishing 🤍🩵,2026,1005,TITLE: rubenylolo......carpfishing 🤍🩵\n\nDESCR...
1,KYkoHio90Pw,Carpfishing 🥵 🔥Summer Campaign 🔥🥵 HOT FISHING🎣,2026,1715,TITLE: Carpfishing 🥵 🔥Summer Campaign 🔥🥵 HOT F...
2,TvT1jlG9ayw,El embalse de Valmayor y su entorno,2016,2537,TITLE: El embalse de Valmayor y su entorno\n\n...
3,kt78DbVibg0,¡LLUVIA y MUCHAS CAPTURAS! 💥 Carp Fishing bajo...,2025,1464,TITLE: ¡LLUVIA y MUCHAS CAPTURAS! 💥 Carp Fishi...
4,dZfMo3t8TWo,🔥Sesion increíble!! 🔥en solitario en valma con...,2025,1684,TITLE: 🔥Sesion increíble!! 🔥en solitario en va...


## Cobertura del conjunto antes de la extracción

Antes de lanzar los lotes, se revisa la distribución temporal y el volumen de vídeos disponibles. Esta comprobación permite asegurar que la extracción no se concentra únicamente en unos pocos años o trimestres.

In [6]:
print("Distribución por año:")
display(df["year"].value_counts().sort_index())

print("\nDistribución por trimestre:")
display(df["year_quarter"].value_counts().sort_index())

Distribución por año:


,count
year,
2009,3
2010,3
2011,7
2012,4
2013,7
2014,10
2015,2
2016,3
2017,7



Distribución por trimestre:


,count
year_quarter,
2009-Q3,2
2009-Q4,1
2010-Q1,1
2010-Q2,2
2011-Q1,1
2011-Q2,1
2011-Q3,3
2011-Q4,2
2012-Q2,1


## Subconjunto operativo para extracción LLM

A partir de `candidates_priority` se define un subconjunto operativo de tamaño controlado para la extracción con LLM. En esta iteración se seleccionan los 151 vídeos con mayor prioridad, y se reparten en tres lotes grandes para reducir el número de iteraciones manuales.

In [7]:
LLM_TOP_K = 151

df_llm_input = (
    df.sort_values(["priority_score"], ascending=False)
      .head(LLM_TOP_K)
      .copy()
      .reset_index(drop=True)
)

print("Shape original:", df.shape)
print("Shape para LLM:", df_llm_input.shape)

cols_show = ["video_id", "title", "year", "year_quarter", "priority_score"]
cols_show = [c for c in cols_show if c in df_llm_input.columns]
df_llm_input[cols_show].head(10)

Shape original: (151, 40)
Shape para LLM: (151, 40)


,video_id,title,year,year_quarter,priority_score
0,K2oTCf-5C4c,rubenylolo......carpfishing 🤍🩵,2026,2026-Q2,21
1,KYkoHio90Pw,Carpfishing 🥵 🔥Summer Campaign 🔥🥵 HOT FISHING🎣,2026,2026-Q1,18
2,TvT1jlG9ayw,El embalse de Valmayor y su entorno,2016,2016-Q3,18
3,kt78DbVibg0,¡LLUVIA y MUCHAS CAPTURAS! 💥 Carp Fishing bajo...,2025,2025-Q4,18
4,dZfMo3t8TWo,🔥Sesion increíble!! 🔥en solitario en valma con...,2025,2025-Q4,18
5,uTM2DGKH-xw,Review Teflon Spare Spools [MV Spools],2021,2021-Q2,16
6,Yt-FrYfIfm4,rubenylolo.....valmayor,2026,2026-Q1,16
7,97L3xUDkpxQ,2ª Jornada de Carpfishing Valmayor,2014,2014-Q2,16
8,DrBaZj0cvLw,rubenylolo......by spinnigcarp,2026,2026-Q1,16
9,5daoPKGBVpI,Jornada de CARPFISHING VALMAYOR con mas de 20 ...,2015,2015-Q1,16


## Asignación de lotes

Para simplificar la extracción semimanual, el subconjunto operativo se divide en tres lotes de tamaño casi uniforme. Cada lote se procesará de forma independiente, pero todos reutilizan la misma lógica de prompt, parseo y guardado.

In [9]:
N_BATCHES = 3

df_llm_input = df_llm_input.copy()
df_llm_input["batch_id"] = np.floor(np.arange(len(df_llm_input)) * N_BATCHES / len(df_llm_input)).astype(int)

print("Número de vídeos:", len(df_llm_input))
print("Número de lotes:", df_llm_input["batch_id"].nunique())
print("\nTamaño por lote:")
display(df_llm_input["batch_id"].value_counts().sort_index())

Número de vídeos: 151
Número de lotes: 3

Tamaño por lote:


,count
batch_id,
0,51
1,50
2,50


## Construcción de un prompt único por lote

En lugar de generar un prompt independiente por vídeo, se construye un único prompt para cada lote. Así el LLM devuelve una única lista JSON con todas las salidas del lote, reduciendo el trabajo manual y manteniendo la trazabilidad.

In [10]:
def build_batch_prompt(df_batch: pd.DataFrame) -> str:
    """
    Construye un único prompt para todo el lote.
    El LLM debe devolver una lista JSON con un objeto por vídeo.
    """
    intro = """
Eres un asistente que analiza metadatos textuales de vídeos de YouTube sobre actividad de pesca.

Debes leer los textos de varios vídeos y devolver únicamente una lista JSON válida, sin explicaciones adicionales.

Para cada vídeo debes identificar, si aparece de forma explícita o razonablemente clara en el texto:

1. si hay actividad de pesca observable o descrita;
2. si se mencionan capturas;
3. qué especie o modalidad aparece;
4. un nivel cualitativo de actividad:
   - "low"
   - "medium"
   - "high"
5. un nivel de certeza:
   - "low"
   - "medium"
   - "high"
6. una evidencia textual breve.

Reglas:
- No inventes capturas ni actividad si no aparecen.
- Si no se puede saber, usa null o listas vacías.
- "species_detected_llm" debe ser una lista de strings.
- "captures_count_llm" debe ser entero o null.
- "activity_mentions_llm" debe ser 1 si hay señal clara de actividad pesquera y 0 si no la hay.
- "activity_level_llm" debe ser uno de: "low", "medium", "high".
- "certainty_llm" debe ser uno de: "low", "medium", "high".
- "evidence_llm" debe ser una frase breve extraída o resumida del texto.
- "notes_llm" puede usarse para anotar ambigüedad.
- Devuelve exactamente una lista JSON con un objeto por cada vídeo del lote.
""".strip()

    output_example = """
Devuelve exactamente una lista JSON con este formato:

[
  {
    "video_id": "xxx",
    "activity_mentions_llm": 0,
    "captures_count_llm": null,
    "species_detected_llm": [],
    "activity_level_llm": "low",
    "certainty_llm": "low",
    "evidence_llm": "",
    "notes_llm": ""
  }
]
""".strip()

    blocks = []
    for _, row in df_batch.iterrows():
        block = f"""
VIDEO
- video_id: {row.get('video_id', '')}
- year: {row.get('year', '')}
- year_quarter: {row.get('year_quarter', '')}
- channel: {row.get('channel', '')}
- title: {row.get('title', '')}

TEXT
{row.get('text_for_llm', '')}
""".strip()
        blocks.append(block)

    return intro + "\n\n" + output_example + "\n\n" + "\n\n--------------------\n\n".join(blocks)

## Selección y generación de un lote

Las siguientes celdas están pensadas para reutilizarse cambiando únicamente el valor de `batch_id_to_process`. Para cada lote se genera un único prompt largo, se pega la respuesta JSON del LLM y se guarda el resultado parseado.

In [41]:
batch_id_to_process = 2

df_batch = (
    df_llm_input[df_llm_input["batch_id"] == batch_id_to_process]
    .copy()
    .reset_index(drop=True)
)

batch_prompt = build_batch_prompt(df_batch)

print("Batch seleccionado:", batch_id_to_process)
print("Vídeos en lote:", len(df_batch))
print("\n--- INICIO DEL PROMPT DEL LOTE ---\n")
len(batch_prompt)


print(batch_prompt)

Batch seleccionado: 2
Vídeos en lote: 50

--- INICIO DEL PROMPT DEL LOTE ---

Eres un asistente que analiza metadatos textuales de vídeos de YouTube sobre actividad de pesca.

Debes leer los textos de varios vídeos y devolver únicamente una lista JSON válida, sin explicaciones adicionales.

Para cada vídeo debes identificar, si aparece de forma explícita o razonablemente clara en el texto:

1. si hay actividad de pesca observable o descrita;
2. si se mencionan capturas;
3. qué especie o modalidad aparece;
4. un nivel cualitativo de actividad:
   - "low"
   - "medium"
   - "high"
5. un nivel de certeza:
   - "low"
   - "medium"
   - "high"
6. una evidencia textual breve.

Reglas:
- No inventes capturas ni actividad si no aparecen.
- Si no se puede saber, usa null o listas vacías.
- "species_detected_llm" debe ser una lista de strings.
- "captures_count_llm" debe ser entero o null.
- "activity_mentions_llm" debe ser 1 si hay señal clara de actividad pesquera y 0 si no la hay.
- "activity

## Pegado de la respuesta del LLM

En esta celda se pega la respuesta completa del LLM correspondiente al lote actual. La respuesta debe ser una lista JSON con una entrada por vídeo del lote.

In [42]:
raw_llm_batch_response = """
[
{
"video_id": "wsPZi-pGDoU",
"activity_mentions_llm": 1,
"captures_count_llm": 1,
"species_detected_llm": ["lucioperca"],
"activity_level_llm": "medium",
"certainty_llm": "high",
"evidence_llm": "Embalse de Valmayor. lucioperca 750g",
"notes_llm": "Captura explícita de lucioperca por título y descripción."
},
{
"video_id": "OGu8eX96vB8",
"activity_mentions_llm": 1,
"captures_count_llm": null,
"species_detected_llm": ["carpa", "lucio", "lucioperca"],
"activity_level_llm": "low",
"certainty_llm": "medium",
"evidence_llm": "Pesca en pato (carpa, lucio y lucioperca)",
"notes_llm": "Actividad de pesca clara por título, pero sin capturas concretas."
},
{
"video_id": "Aay5eNQqok",
"activity_mentions_llm": 1,
"captures_count_llm": null,
"species_detected_llm": ["carpa"],
"activity_level_llm": "medium",
"certainty_llm": "high",
"evidence_llm": "We catch a few carp",
"notes_llm": "Capturas claras en lagos de Madrid incluyendo Valmayor, pero sin número exacto."
},
{
"video_id": "J6J6g2nzG_k",
"activity_mentions_llm": 1,
"captures_count_llm": 20,
"species_detected_llm": ["carpa"],
"activity_level_llm": "high",
"certainty_llm": "high",
"evidence_llm": "pudimos sacar mas de 20 peces",
"notes_llm": "Se usa 20 como mínimo explícito; el lugar indicado es El Vellón."
},
{
"video_id": "OVOVS93NQco",
"activity_mentions_llm": 1,
"captures_count_llm": null,
"species_detected_llm": ["carpa"],
"activity_level_llm": "medium",
"certainty_llm": "high",
"evidence_llm": "me consigo hacer con unas carpitas",
"notes_llm": "Capturas claras, pero sin número exacto."
},
{
"video_id": "KFC3Zatbs54",
"activity_mentions_llm": 1,
"captures_count_llm": null,
"species_detected_llm": ["carpa"],
"activity_level_llm": "high",
"certainty_llm": "high",
"evidence_llm": "conseguimos sacar un monton de carpas durante el dia",
"notes_llm": "Capturas claras, pero sin número exacto; el lugar indicado es El Vellón."
},
{
"video_id": "apw2MmnMDeA",
"activity_mentions_llm": 1,
"captures_count_llm": null,
"species_detected_llm": ["carpa"],
"activity_level_llm": "low",
"certainty_llm": "low",
"evidence_llm": "Carpfishing///Mcarp-Winter in Valmayor",
"notes_llm": "Solo hay título; actividad de carpfishing clara, sin capturas concretas."
},
{
"video_id": "8YQiu5llziI",
"activity_mentions_llm": 1,
"captures_count_llm": null,
"species_detected_llm": ["carpa"],
"activity_level_llm": "medium",
"certainty_llm": "high",
"evidence_llm": "disfrutar de unas cuantas carpitas de valmayor",
"notes_llm": "Capturas claras, pero sin número exacto."
},
{
"video_id": "0-nRTYbPDSE",
"activity_mentions_llm": 1,
"captures_count_llm": null,
"species_detected_llm": ["carpa"],
"activity_level_llm": "low",
"certainty_llm": "medium",
"evidence_llm": "otra sesion entretenida en la que pudimos disfrutar de lo que nos gusta",
"notes_llm": "Actividad de carpfishing clara por título, pero sin capturas concretas."
},
{
"video_id": "1RdIha9GsXU",
"activity_mentions_llm": 1,
"captures_count_llm": null,
"species_detected_llm": ["carpa"],
"activity_level_llm": "low",
"certainty_llm": "medium",
"evidence_llm": "Video testing en Valmayor sobre la marca inglesa ATOMIC",
"notes_llm": "Carpfishing en Valmayor, pero parece prueba de material y no se mencionan capturas."
},
{
"video_id": "3XzMcvagakI",
"activity_mentions_llm": 1,
"captures_count_llm": 4,
"species_detected_llm": ["carpa"],
"activity_level_llm": "medium",
"certainty_llm": "high",
"evidence_llm": "hacernos con 4 carpas simultaneas",
"notes_llm": "Capturas explícitas; el lugar descrito es El Vellón."
},
{
"video_id": "2ZTNypKqAfk",
"activity_mentions_llm": 1,
"captures_count_llm": null,
"species_detected_llm": ["carpa"],
"activity_level_llm": "high",
"certainty_llm": "medium",
"evidence_llm": "Dia de lluvia en El Vellon con un no parar de carpas",
"notes_llm": "El título indica muchas carpas, pero sin número exacto."
},
{
"video_id": "nBs34O9G2vA",
"activity_mentions_llm": 1,
"captures_count_llm": 1,
"species_detected_llm": ["carpa royal"],
"activity_level_llm": "medium",
"certainty_llm": "high",
"evidence_llm": "bonita carpa royal sacada por pedro en valmayor",
"notes_llm": "Captura explícita de carpa royal de 7 kg."
},
{
"video_id": "5yAOCx4-Jo",
"activity_mentions_llm": 1,
"captures_count_llm": null,
"species_detected_llm": ["carpa"],
"activity_level_llm": "low",
"certainty_llm": "medium",
"evidence_llm": "nuestra ultima sesion en Valmayor",
"notes_llm": "Actividad de carpfishing clara, sin capturas concretas."
},
{
"video_id": "2ZvYjrkBNDg",
"activity_mentions_llm": 1,
"captures_count_llm": null,
"species_detected_llm": ["carpa"],
"activity_level_llm": "low",
"certainty_llm": "medium",
"evidence_llm": "CARPFISHING EN VALMAYOR",
"notes_llm": "Actividad clara por título y descripción, sin capturas concretas."
},
{
"video_id": "mF92dteqKEQ",
"activity_mentions_llm": 1,
"captures_count_llm": null,
"species_detected_llm": ["carpa"],
"activity_level_llm": "medium",
"certainty_llm": "medium",
"evidence_llm": "el resultado fue satisfactorio",
"notes_llm": "Sesión de carpfishing clara y resultado positivo, pero no se especifican capturas."
},
{
"video_id": "iORT45J9Kkw",
"activity_mentions_llm": 1,
"captures_count_llm": null,
"species_detected_llm": ["carpa"],
"activity_level_llm": "low",
"certainty_llm": "medium",
"evidence_llm": "Sesión express en el pantano de valmayor",
"notes_llm": "Actividad de carpfishing clara, sin capturas concretas."
},
{
"video_id": "feZ13jeezAk",
"activity_mentions_llm": 1,
"captures_count_llm": null,
"species_detected_llm": ["carpa"],
"activity_level_llm": "low",
"certainty_llm": "low",
"evidence_llm": "carpfishing en valmayor",
"notes_llm": "Solo hay título; actividad clara, sin capturas."
},
{
"video_id": "M-3nBo79IFI",
"activity_mentions_llm": 1,
"captures_count_llm": null,
"species_detected_llm": ["carpa"],
"activity_level_llm": "low",
"certainty_llm": "low",
"evidence_llm": "Extreme carpfishing in Spain, Valmayor",
"notes_llm": "Solo hay título; actividad clara, sin capturas concretas."
},
{
"video_id": "UTT4FzK33E4",
"activity_mentions_llm": 1,
"captures_count_llm": 1,
"species_detected_llm": ["black bass"],
"activity_level_llm": "medium",
"certainty_llm": "high",
"evidence_llm": "Gran Black bass a superficie con el poper",
"notes_llm": "Captura explícita de black bass; no se menciona Valmayor en el texto."
},
{
"video_id": "uyrqNk9vgFM",
"activity_mentions_llm": 1,
"captures_count_llm": null,
"species_detected_llm": ["carpa"],
"activity_level_llm": "low",
"certainty_llm": "low",
"evidence_llm": "CARPFISHING en VALMAYOR",
"notes_llm": "Solo hay título; actividad clara, sin capturas concretas."
},
{
"video_id": "Kh32Oek6ekY",
"activity_mentions_llm": 1,
"captures_count_llm": 1,
"species_detected_llm": ["carpa"],
"activity_level_llm": "medium",
"certainty_llm": "high",
"evidence_llm": "Mi primera carpa tras el confinamiento",
"notes_llm": "Captura explícita de una carpa."
},
{
"video_id": "qpn-LUY19sE",
"activity_mentions_llm": 1,
"captures_count_llm": 0,
"species_detected_llm": ["carpa"],
"activity_level_llm": "low",
"certainty_llm": "high",
"evidence_llm": "Sesión caracterizada por la ausencia de picadas",
"notes_llm": "Actividad de pesca clara, pero sin picadas ni capturas."
},
{
"video_id": "3C8GUVfMSTk",
"activity_mentions_llm": 0,
"captures_count_llm": null,
"species_detected_llm": [],
"activity_level_llm": "low",
"certainty_llm": "medium",
"evidence_llm": "Dentro de poquito tendremos nuevos videos en el canal",
"notes_llm": "Anuncio genérico del canal, sin actividad de pesca observable ni capturas."
},
{
"video_id": "VbVkFWjz9Pg",
"activity_mentions_llm": 1,
"captures_count_llm": 5,
"species_detected_llm": ["barbo", "carpa", "lucio", "lucioperca"],
"activity_level_llm": "high",
"certainty_llm": "high",
"evidence_llm": "salieron una gran carpa, dos lucios, una lucioperca y este bonito barbo",
"notes_llm": "Capturas explícitas; el título indica Buendía, no Valmayor."
},
{
"video_id": "kMwR2j-xT-A",
"activity_mentions_llm": 1,
"captures_count_llm": null,
"species_detected_llm": ["carpa"],
"activity_level_llm": "low",
"certainty_llm": "low",
"evidence_llm": "Carpfishing Madrid, Valmayor",
"notes_llm": "Título indica carpfishing en Valmayor, pero la descripción solo promociona una app."
},
{
"video_id": "qc7X3EXe3Po",
"activity_mentions_llm": 1,
"captures_count_llm": null,
"species_detected_llm": ["carpa"],
"activity_level_llm": "low",
"certainty_llm": "low",
"evidence_llm": "carpa valmayor club fuenticarp",
"notes_llm": "Título/descripción indican carpa en Valmayor, pero sin detalle de captura."
},
{
"video_id": "CsCaKLt8LWE",
"activity_mentions_llm": 1,
"captures_count_llm": null,
"species_detected_llm": ["carpa"],
"activity_level_llm": "low",
"certainty_llm": "medium",
"evidence_llm": "Sesión de unas horas en valmayor",
"notes_llm": "Actividad de carpfishing clara, sin capturas concretas."
},
{
"video_id": "1IKeEjnYtRc",
"activity_mentions_llm": 1,
"captures_count_llm": null,
"species_detected_llm": ["carpa"],
"activity_level_llm": "medium",
"certainty_llm": "low",
"evidence_llm": "Fantástico día de carpfishing en valmayor",
"notes_llm": "Solo hay título; actividad clara, pero no se confirman capturas."
},
{
"video_id": "3Vb3tD5AuVU",
"activity_mentions_llm": 1,
"captures_count_llm": 0,
"species_detected_llm": ["carpa"],
"activity_level_llm": "low",
"certainty_llm": "high",
"evidence_llm": "los peces no ayudaron",
"notes_llm": "El título indica bolo y la descripción confirma ausencia de resultados."
},
{
"video_id": "poyAb2k6y8o",
"activity_mentions_llm": 1,
"captures_count_llm": null,
"species_detected_llm": ["carpa"],
"activity_level_llm": "low",
"certainty_llm": "medium",
"evidence_llm": "buen día pasamos muy entretenido",
"notes_llm": "Actividad de carpfishing clara por título, sin capturas concretas."
},
{
"video_id": "uaU9rVoMOWg",
"activity_mentions_llm": 1,
"captures_count_llm": 1,
"species_detected_llm": ["carpa"],
"activity_level_llm": "medium",
"certainty_llm": "high",
"evidence_llm": "segunda parte de la pelea con una carpa en valmayor",
"notes_llm": "Captura/lucha con carpa explícita."
},
{
"video_id": "go0kMD7ryu4",
"activity_mentions_llm": 1,
"captures_count_llm": null,
"species_detected_llm": ["carpa"],
"activity_level_llm": "low",
"certainty_llm": "low",
"evidence_llm": "Carpfishing Valmayor, carpas y mucho calor",
"notes_llm": "Solo hay título; actividad clara, sin capturas cuantificadas."
},
{
"video_id": "oQTfdfSQwME",
"activity_mentions_llm": 1,
"captures_count_llm": 1,
"species_detected_llm": ["carpa"],
"activity_level_llm": "medium",
"certainty_llm": "medium",
"evidence_llm": "carpa de Valmayor",
"notes_llm": "Solo hay título; indica una carpa, probablemente captura."
},
{
"video_id": "bZtvMfc65bk",
"activity_mentions_llm": 1,
"captures_count_llm": 1,
"species_detected_llm": ["carpa"],
"activity_level_llm": "medium",
"certainty_llm": "medium",
"evidence_llm": "Carpa en valmayor",
"notes_llm": "Solo hay título; indica una carpa, probablemente captura."
},
{
"video_id": "jECSWpkRUj4",
"activity_mentions_llm": 1,
"captures_count_llm": null,
"species_detected_llm": ["carpa"],
"activity_level_llm": "low",
"certainty_llm": "medium",
"evidence_llm": "Valmayor concurso",
"notes_llm": "Concurso de carpfishing en Valmayor, sin capturas concretas."
},
{
"video_id": "eeJDivixAss",
"activity_mentions_llm": 1,
"captures_count_llm": null,
"species_detected_llm": ["carpa"],
"activity_level_llm": "low",
"certainty_llm": "low",
"evidence_llm": "Carpfishing Valmayor 2021",
"notes_llm": "Solo hay título; actividad clara, sin capturas."
},
{
"video_id": "xFJGP-WJpuA",
"activity_mentions_llm": 1,
"captures_count_llm": null,
"species_detected_llm": ["carpa"],
"activity_level_llm": "medium",
"certainty_llm": "low",
"evidence_llm": "carpfishing valmayor casi doblete",
"notes_llm": "El título sugiere actividad y casi doblete, pero no confirma número de capturas."
},
{
"video_id": "4USupq8rqEI",
"activity_mentions_llm": 1,
"captures_count_llm": 1,
"species_detected_llm": ["carpa"],
"activity_level_llm": "medium",
"certainty_llm": "high",
"evidence_llm": "Carpa de 11 kilos Valmayor",
"notes_llm": "Captura explícita por título."
},
{
"video_id": "AEFOVROYVHE",
"activity_mentions_llm": 1,
"captures_count_llm": null,
"species_detected_llm": ["carpa"],
"activity_level_llm": "low",
"certainty_llm": "low",
"evidence_llm": "CARPFISHING en valmayor",
"notes_llm": "Solo hay título; actividad clara, sin capturas."
},
{
"video_id": "3_7HQoMoU1I",
"activity_mentions_llm": 1,
"captures_count_llm": 1,
"species_detected_llm": ["carpa"],
"activity_level_llm": "medium",
"certainty_llm": "medium",
"evidence_llm": "carpa mui luchadora de buen porte",
"notes_llm": "Título indica una carpa capturada/luchadora, sin descripción adicional."
},
{
"video_id": "yJQyMewYnSM",
"activity_mentions_llm": 0,
"captures_count_llm": null,
"species_detected_llm": [],
"activity_level_llm": "low",
"certainty_llm": "low",
"evidence_llm": "memories of valmayor",
"notes_llm": "Título demasiado genérico; no hay señal suficiente de pesca o capturas."
},
{
"video_id": "ys9Hd3B1cP0",
"activity_mentions_llm": 0,
"captures_count_llm": null,
"species_detected_llm": ["carpa"],
"activity_level_llm": "low",
"certainty_llm": "high",
"evidence_llm": "2 TÉCNICAS COMO USAR EL MAÍZ PARA PESCAR CARPAS",
"notes_llm": "Vídeo tutorial de cebo/técnica, no jornada de pesca observable ni capturas concretas."
},
{
"video_id": "LvFq0OOQ3Y8",
"activity_mentions_llm": 0,
"captures_count_llm": null,
"species_detected_llm": ["carpa"],
"activity_level_llm": "low",
"certainty_llm": "high",
"evidence_llm": "la mejor carnada 100% efectiva para pescar carpas",
"notes_llm": "Vídeo tutorial de carnada, no describe una captura concreta."
},
{
"video_id": "NgQOFljSa40",
"activity_mentions_llm": 1,
"captures_count_llm": null,
"species_detected_llm": ["carpa"],
"activity_level_llm": "medium",
"certainty_llm": "medium",
"evidence_llm": "una mañana de pesca con mucha accion en medio de una gran lluvia",
"notes_llm": "Actividad clara y mucha acción, pero sin capturas explícitas ni Valmayor."
},
{
"video_id": "md4HFLhF2qY",
"activity_mentions_llm": 0,
"captures_count_llm": null,
"species_detected_llm": [],
"activity_level_llm": "low",
"certainty_llm": "high",
"evidence_llm": "Dos chicas de 13 y 14 años han fallecido hoy ahogadas",
"notes_llm": "Vídeo de noticia/sucesos, no actividad de pesca."
},
{
"video_id": "K6n_0CwMTos",
"activity_mentions_llm": 1,
"captures_count_llm": null,
"species_detected_llm": ["lucio"],
"activity_level_llm": "low",
"certainty_llm": "low",
"evidence_llm": "Valmayor LUCIO",
"notes_llm": "Solo hay título; actividad de pesca probable, sin captura confirmada."
},
{
"video_id": "hz5Dr7Y1BSk",
"activity_mentions_llm": 0,
"captures_count_llm": null,
"species_detected_llm": [],
"activity_level_llm": "low",
"certainty_llm": "high",
"evidence_llm": "Embalses de Madrid: Pesca, Naturaleza y Ocio",
"notes_llm": "Vídeo informativo general sobre embalses; no describe una jornada de pesca ni capturas."
},
{
"video_id": "1GkZuB7OLxg",
"activity_mentions_llm": 1,
"captures_count_llm": 1,
"species_detected_llm": ["lucio"],
"activity_level_llm": "medium",
"certainty_llm": "high",
"evidence_llm": "hemos sacado este bonito lucio",
"notes_llm": "Captura explícita de lucio."
},
{
"video_id": "C1hH10HyTt4",
"activity_mentions_llm": 1,
"captures_count_llm": 1,
"species_detected_llm": ["lucio"],
"activity_level_llm": "medium",
"certainty_llm": "high",
"evidence_llm": "Otro bonito lucio del pantano Valmayor",
"notes_llm": "Captura explícita de lucio."
}
]
"""

In [43]:
def parse_llm_response(raw_text: str) -> list:
    """
    Parsea una respuesta JSON del LLM.
    """
    raw_text = raw_text.strip()
    if not raw_text:
        return []

    try:
        data = json.loads(raw_text)
    except Exception as e:
        print("Error parseando JSON:", e)
        return []

    if isinstance(data, dict):
        data = [data]

    return data

In [44]:
parsed_batch = parse_llm_response(raw_llm_batch_response)
batch_result_df = pd.DataFrame(parsed_batch)

print("Filas parseadas:", len(batch_result_df))
batch_result_df.head()

Filas parseadas: 50


,video_id,activity_mentions_llm,captures_count_llm,species_detected_llm,activity_level_llm,certainty_llm,evidence_llm,notes_llm
0,wsPZi-pGDoU,1,1.0,[lucioperca],medium,high,Embalse de Valmayor. lucioperca 750g,Captura explícita de lucioperca por título y d...
1,OGu8eX96vB8,1,NaN,"[carpa, lucio, lucioperca]",low,medium,"Pesca en pato (carpa, lucio y lucioperca)","Actividad de pesca clara por título, pero sin ..."
2,Aay5eNQqok,1,NaN,[carpa],medium,high,We catch a few carp,Capturas claras en lagos de Madrid incluyendo ...
3,J6J6g2nzG_k,1,20.0,[carpa],high,high,pudimos sacar mas de 20 peces,Se usa 20 como mínimo explícito; el lugar indi...
4,OVOVS93NQco,1,NaN,[carpa],medium,high,me consigo hacer con unas carpitas,"Capturas claras, pero sin número exacto."


In [45]:
print("Vídeos esperados en lote:", len(df_batch))

if len(batch_result_df) > 0 and "video_id" in batch_result_df.columns:
    print("Vídeos devueltos:", batch_result_df["video_id"].nunique())
    print("Duplicados en video_id:", batch_result_df["video_id"].duplicated().sum())
    print("\nNulos por columna:")
    display(batch_result_df.isna().sum())
else:
    print("batch_result_df vacío o sin columna video_id")

Vídeos esperados en lote: 50
Vídeos devueltos: 50
Duplicados en video_id: 0

Nulos por columna:


,0
video_id,0
activity_mentions_llm,0
captures_count_llm,34
species_detected_llm,0
activity_level_llm,0
certainty_llm,0
evidence_llm,0
notes_llm,0


## Guardado intermedio del lote

Cada lote procesado se guarda por separado en formato Parquet. Esto permite reanudar el proceso sin repetir lotes ya completados y facilita la consolidación posterior.

In [46]:
batch_output_path = OUTPUT_DIR / f"llm_priority_batch_{batch_id_to_process:03d}.parquet"

if len(batch_result_df) > 0:
    batch_result_df.to_parquet(batch_output_path, index=False)
    print("Guardado lote en:", batch_output_path)
else:
    print("No se guarda nada porque batch_result_df está vacío.")

Guardado lote en: /content/drive/MyDrive/PIDS4jjj2/outputs/llm_activity/llm_priority_batch_002.parquet


## Consolidación de lotes procesados

Una vez procesados y guardados los lotes individuales, en este bloque se cargan todos los ficheros intermedios, se concatenan en un único `DataFrame` y se realizan comprobaciones finales de integridad.

El objetivo es obtener un dataset consolidado con una fila por vídeo, listo para el análisis posterior.

In [47]:
from pathlib import Path
import pandas as pd

batch_files = sorted(OUTPUT_DIR.glob("llm_priority_batch_*.parquet"))

print("Número de ficheros de lote encontrados:", len(batch_files))
for f in batch_files:
    print("-", f.name)

Número de ficheros de lote encontrados: 3
- llm_priority_batch_000.parquet
- llm_priority_batch_001.parquet
- llm_priority_batch_002.parquet


In [48]:
dfs = [pd.read_parquet(f) for f in batch_files]

df_llm_all = pd.concat(dfs, ignore_index=True)

print("Shape consolidado:", df_llm_all.shape)
df_llm_all.head()

Shape consolidado: (151, 8)


,video_id,activity_mentions_llm,captures_count_llm,species_detected_llm,activity_level_llm,certainty_llm,evidence_llm,notes_llm
0,K2oTCf-5C4c,1,NaN,"[carpa, barbo, lucio, bass]",low,medium,salidas de pesca en varias modalidades y sobre...,"Se menciona actividad de pesca y modalidades, ..."
1,KYkoHio90Pw,1,NaN,"[carpa, lucioperca]",high,high,Sorpresas con peces grandes en lagos urbanos d...,Hay actividad pesquera clara y capturas mencio...
2,TvT1jlG9ayw,1,NaN,"[lucio, carpa]",low,medium,"se pueden recorrer en bici o a pie, hacer pira...",Vídeo descriptivo del entorno; menciona pesca ...
3,kt78DbVibg0,1,NaN,[carpa],high,high,LLUVIA y MUCHAS CAPTURAS; las carpas dieron la...,"Capturas claras, pero sin número exacto."
4,dZfMo3t8TWo,1,NaN,[carpa],high,high,sesión increíble en solitario en valma con dob...,"Se mencionan picadas y carpas, pero no número ..."


## Validación del consolidado

Se comprueba que el dataset consolidado:
- no tenga duplicados por `video_id`,
- no haya perdido vídeos respecto al conjunto de entrada del notebook,
- y mantenga una estructura coherente para el resto del pipeline.

In [49]:
print("Vídeos consolidados:", len(df_llm_all))
print("IDs únicos:", df_llm_all["video_id"].nunique())
print("Duplicados por video_id:", int(df_llm_all["video_id"].duplicated().sum()))
print("Nulos por columna:")
display(df_llm_all.isna().sum().sort_values(ascending=False))

Vídeos consolidados: 151
IDs únicos: 151
Duplicados por video_id: 0
Nulos por columna:


,0
captures_count_llm,99
video_id,0
activity_mentions_llm,0
species_detected_llm,0
activity_level_llm,0
certainty_llm,0
evidence_llm,0
notes_llm,0


In [51]:
df_llm_all = df_llm_all.drop_duplicates(subset=["video_id"]).reset_index(drop=True)

print("Shape tras drop_duplicates:", df_llm_all.shape)

Shape tras drop_duplicates: (151, 8)


## Guardado del dataset consolidado

El resultado final de la extracción LLM se persiste en formato Parquet y CSV para facilitar tanto el análisis posterior como la inspección manual.

In [52]:
final_parquet_path = OUTPUT_DIR / "llm_priority_dataset.parquet"
final_csv_path = OUTPUT_DIR / "llm_priority_dataset.csv"

df_llm_all.to_parquet(final_parquet_path, index=False)
df_llm_all.to_csv(final_csv_path, index=False)

print("Guardado parquet final:", final_parquet_path)
print("Guardado csv final:", final_csv_path)
print("Shape final:", df_llm_all.shape)

Guardado parquet final: /content/drive/MyDrive/PIDS4jjj2/outputs/llm_activity/llm_priority_dataset.parquet
Guardado csv final: /content/drive/MyDrive/PIDS4jjj2/outputs/llm_activity/llm_priority_dataset.csv
Shape final: (151, 8)


In [53]:
df_check = pd.read_parquet(final_parquet_path)

print("Shape recargado:", df_check.shape)
df_check.head()

Shape recargado: (151, 8)


,video_id,activity_mentions_llm,captures_count_llm,species_detected_llm,activity_level_llm,certainty_llm,evidence_llm,notes_llm
0,K2oTCf-5C4c,1,NaN,"[carpa, barbo, lucio, bass]",low,medium,salidas de pesca en varias modalidades y sobre...,"Se menciona actividad de pesca y modalidades, ..."
1,KYkoHio90Pw,1,NaN,"[carpa, lucioperca]",high,high,Sorpresas con peces grandes en lagos urbanos d...,Hay actividad pesquera clara y capturas mencio...
2,TvT1jlG9ayw,1,NaN,"[lucio, carpa]",low,medium,"se pueden recorrer en bici o a pie, hacer pira...",Vídeo descriptivo del entorno; menciona pesca ...
3,kt78DbVibg0,1,NaN,[carpa],high,high,LLUVIA y MUCHAS CAPTURAS; las carpas dieron la...,"Capturas claras, pero sin número exacto."
4,dZfMo3t8TWo,1,NaN,[carpa],high,high,sesión increíble en solitario en valma con dob...,"Se mencionan picadas y carpas, pero no número ..."


In [54]:
print("Distribución activity_mentions_llm:")
display(df_llm_all["activity_mentions_llm"].value_counts(dropna=False))

print("\nDistribución activity_level_llm:")
display(df_llm_all["activity_level_llm"].value_counts(dropna=False))

print("\nDistribución certainty_llm:")
display(df_llm_all["certainty_llm"].value_counts(dropna=False))

Distribución activity_mentions_llm:


,count
activity_mentions_llm,
1,142
0,9



Distribución activity_level_llm:


,count
activity_level_llm,
medium,65
low,59
high,27



Distribución certainty_llm:


,count
certainty_llm,
high,83
medium,51
low,17


In [56]:
df_species_preview = df_llm_all[["video_id", "species_detected_llm"]].head(15)
df_species_preview

,video_id,species_detected_llm
0,K2oTCf-5C4c,"[carpa, barbo, lucio, bass]"
1,KYkoHio90Pw,"[carpa, lucioperca]"
2,TvT1jlG9ayw,"[lucio, carpa]"
3,kt78DbVibg0,[carpa]
4,dZfMo3t8TWo,[carpa]
5,uTM2DGKH-xw,[]
6,Yt-FrYfIfm4,[]
7,97L3xUDkpxQ,[carpa]
8,DrBaZj0cvLw,[]
9,5daoPKGBVpI,[carpa]


## Conclusión

Este notebook deja consolidada la extracción semiestructurada realizada con LLM sobre el subconjunto priorizado de vídeos. El resultado es un dataset intermedio con variables interpretables por vídeo, como presencia de actividad pesquera, número de capturas cuando aparece de forma explícita, especies detectadas y nivel cualitativo de actividad.

Este fichero consolidado servirá como base para el análisis posterior y para la construcción de la variable objetivo en los siguientes notebooks del pipeline.